# Exploratory analysis

Run generator and database loader first. This notebook is intentionally small; dashboard and reusable modules hold production logic.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
with sqlite3.connect(ROOT / 'data/network_metrics.db') as conn:
    df = pd.read_sql_query('SELECT * FROM device_telemetry', conn, parse_dates=['timestamp'])
df.head()

In [ ]:
df.assign(utilization_pct=100 * df.bandwidth_usage_mbps / df.bandwidth_capacity_mbps).groupby('site_id').agg(
    uptime_sla_pct=('uptime_status', lambda x: 100 * x.mean()),
    avg_latency_ms=('latency_ms', 'mean'),
    avg_packet_loss_pct=('packet_loss_pct', 'mean'),
    incident_count=('is_incident', 'sum')
).round(2).sort_values('uptime_sla_pct')